# Context Compaction in Dobby — Live Demo

As an AI agent works, every tool call adds its result to the conversation. Those results pile up and get **re-sent to the model on every turn**. The cost is real:

- 💸 **Money** — you pay for the same tokens again and again
- 🐢 **Latency** — bigger prompts are slower
- 🧱 **The wall** — eventually you overflow the model's context window and the agent breaks

**Dobby compacts the context automatically, mid-run.** This notebook drives a real research agent against a live model and shows the context staying small *while the agent keeps working*. Every number below is produced by the run — nothing is hard-coded.

## Two strategies

| | **Trim** | **Summarize** |
|---|---|---|
| What it does | Replaces stale tool results with a short placeholder | Collapses an old span into one LLM-written digest |
| Cost | Free — pure text substitution | One LLM call per compaction |
| Reversible? | Yes — originals are kept on the edit record | No — it is a write-back |
| Best for | Bounding cost on tool-heavy runs | Keeping the *meaning* of old context |

Both are driven by one `ContextPolicy` and fire automatically when context crosses a threshold. You can also let the agent compact itself with a tool (Step 4).

In [1]:
from dataclasses import dataclass, field
import os
from typing import Annotated, Any

from dotenv import load_dotenv

from dobby import AgentExecutor, CompactContextTool, ContextEditEvent, ContextPolicy
from dobby.context import edit_context
from dobby.context._tokens import estimate_input_tokens
from dobby.providers.openai import OpenAIProvider
from dobby.tools import Tool
from dobby.types import (
    AssistantMessagePart,
    StreamEndEvent,
    TextPart,
    ToolResultPart,
    ToolUseEvent,
    ToolUsePart,
    UserMessagePart,
)

load_dotenv()


def make_executor(*, tools, context_policy):
    """Build an AgentExecutor, preferring Azure OpenAI when configured."""
    if os.getenv("AZURE_OPENAI_ENDPOINT"):
        provider = OpenAIProvider(
            base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment_id=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        )
        kind = "azure-openai"
    elif os.getenv("OPENAI_API_KEY"):
        provider = OpenAIProvider(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))
        kind = "openai"
    else:
        raise RuntimeError("Set AZURE_OPENAI_ENDPOINT or OPENAI_API_KEY in .env")
    return AgentExecutor(provider=kind, llm=provider, tools=tools, context_policy=context_policy)


PROVIDER = (
    "Azure OpenAI"
    if os.getenv("AZURE_OPENAI_ENDPOINT")
    else ("OpenAI" if os.getenv("OPENAI_API_KEY") else "NONE — set keys in .env")
)
print("Provider:", PROVIDER)

Provider: Azure OpenAI


## The workload: a research agent

Our agent fetches a **detailed report** for five topics — `alpha, beta, gamma, delta, epsilon` — one per turn, then writes a one-line summary of each. Each report is deliberately large, so the context grows fast. Think of it as a stand-in for any real tool-heavy agent (search, file reads, API calls).

In [2]:
@dataclass
class FetchReportTool(Tool):
    name = "fetch_report"
    description = "Fetch a detailed report for a topic. Returns a long document."

    async def __call__(self, topic: Annotated[str, "The report topic"]) -> dict[str, Any]:
        body = " ".join(f"{topic}-finding-{i}: value={i * 7}" for i in range(200))
        return {"topic": topic, "report": body}


TOPICS = ["alpha", "beta", "gamma", "delta", "epsilon"]
MESSAGES = [
    UserMessagePart(
        parts=[
            TextPart(
                text=(
                    "Fetch reports for these topics one at a time, then summarize each "
                    f"in one line: {', '.join(TOPICS)}."
                )
            )
        ]
    )
]

SYSTEM_PROMPT = (
    "You are a research assistant. You MUST fetch a report for EVERY requested topic "
    "before writing any summary. Call fetch_report for exactly ONE topic per turn, and "
    "do not summarize or stop until all topics have been fetched. Never call "
    "fetch_report more than once in a single turn."
)

# How heavy is one report?
sample = await FetchReportTool()(topic="alpha")
one = estimate_input_tokens(
    [
        UserMessagePart(
            parts=[
                ToolResultPart(
                    tool_use_id="x", name="fetch_report", parts=[TextPart(text=sample["report"])]
                )
            ]
        )
    ]
)
print(f"Each report  ≈ {one:,} tokens of context")
print(f"Five reports ≈ {one * len(TOPICS):,} tokens if nothing is compacted")

Each report  ≈ 1,432 tokens of context
Five reports ≈ 7,160 tokens if nothing is compacted


### Instrumentation

A small harness drives the agent and records, for every turn, **how many tokens** were sent to the model and **how many reports** have been fetched so far. The bar charts below are rendered straight from that record.

In [3]:
@dataclass
class RunTrace:
    label: str
    tokens: list[int] = field(default_factory=list)  # input tokens sent each turn
    concepts: list[int] = field(default_factory=list)  # reports fetched so far
    edits: list[ContextEditEvent] = field(default_factory=list)


async def run_agent(label, *, policy, tools=None, system_prompt=SYSTEM_PROMPT, max_iterations=12):
    """Run the agent once and capture a per-turn trace of context size + work done."""
    executor = make_executor(tools=tools or [FetchReportTool()], context_policy=policy)
    trace = RunTrace(label=label)
    fetched = 0
    async for event in executor.run_stream(
        MESSAGES, system_prompt=system_prompt, max_iterations=max_iterations
    ):
        if isinstance(event, ToolUseEvent) and event.name == "fetch_report":
            fetched += 1
        elif isinstance(event, ContextEditEvent):
            trace.edits.append(event)
        elif isinstance(event, StreamEndEvent) and event.usage:
            trace.tokens.append(event.usage.input_tokens)
            trace.concepts.append(fetched)
    return trace


def _bar(value, peak, width=42):
    filled = int(round((value / peak) * width)) if peak else 0
    return "█" * filled + "░" * (width - filled)


def show_trace(trace, peak=None):
    """Print one run as a bar chart: bar = context tokens, number = reports fetched."""
    peak = peak or (max(trace.tokens) if trace.tokens else 1)
    print(f"  {trace.label}")
    print(f"  {'turn':>4} {'reports':>8}  context (tokens)")
    for i, (tok, con) in enumerate(zip(trace.tokens, trace.concepts, strict=False), 1):
        print(f"  {i:>4} {con:>8}  {_bar(tok, peak)} {tok:>7,}")
    if trace.edits:
        cleared = sum((ae.cleared_tool_uses or 0) for e in trace.edits for ae in e.applied_edits)
        print(f"  → {len(trace.edits)} compaction(s); cleared/condensed {cleared} tool result(s)")


def compare(a, b):
    """Render two runs on the same scale so the difference is obvious."""
    peak = max(max(a.tokens), max(b.tokens))
    show_trace(a, peak=peak)
    print()
    show_trace(b, peak=peak)


def show_dashboard(traces):
    """Side-by-side scorecard across strategies."""
    base_peak = max(traces[0].tokens)
    print(
        f"{'strategy':<12}{'reports':>9}{'peak context':>14}{'vs baseline':>13}{'extra LLM calls':>17}"
    )
    print("  " + "-" * 63)
    for t in traces:
        peak = max(t.tokens)
        rpt = max(t.concepts) if t.concepts else 0
        delta = "—" if t is traces[0] else f"{(peak - base_peak) / base_peak * 100:+.0f}%"
        extra = sum(1 for e in t.edits for ae in e.applied_edits if ae.type == "summarize")
        print(f"{t.label:<12}{rpt:>9}{peak:>14,}{delta:>13}{extra:>17}")

## Step 1 — The problem: context grows unbounded

First, the agent runs with **no compaction**. The bar is the context sent to the model each turn; the number is how many reports have been fetched. Watch the bar climb as work piles up.

In [4]:
baseline = await run_agent("baseline", policy=None)
show_trace(baseline)
print(
    f"\nPeak context: {max(baseline.tokens):,} tokens to fetch {max(baseline.concepts)} reports."
)

  baseline
  turn  reports  context (tokens)
     1        1  █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░     140
     2        2  █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   2,035
     3        3  █████████████████░░░░░░░░░░░░░░░░░░░░░░░░░   3,930
     4        4  █████████████████████████░░░░░░░░░░░░░░░░░   5,825
     5        5  ██████████████████████████████████░░░░░░░░   7,720
     6        5  ██████████████████████████████████████████   9,615

Peak context: 9,615 tokens to fetch 5 reports.


## Step 2 — Trim: bound the context for free

Now we attach `ContextPolicy(mode="trim")`. The moment context crosses the threshold, Dobby swaps the **oldest** tool results for a tiny placeholder and keeps the most recent ones intact. Same agent, same five reports — but the context stops growing.

> The window is set to a small `2,000` tokens so compaction fires within a few turns for the demo. In production you'd set this to your model's real window (e.g. `128,000`).

In [5]:
TRIM = ContextPolicy(context_window=2000, trigger_pct=0.5, keep_last_n=2, mode="trim")
trim = await run_agent("trim", policy=TRIM)
compare(baseline, trim)
saved = (max(baseline.tokens) - max(trim.tokens)) / max(baseline.tokens) * 100
print(
    f"\nPeak context: {max(baseline.tokens):,} → {max(trim.tokens):,} tokens "
    f"({saved:.0f}% smaller) — with 0 extra LLM calls."
)

  baseline
  turn  reports  context (tokens)
     1        1  █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░     140
     2        2  █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   2,035
     3        3  █████████████████░░░░░░░░░░░░░░░░░░░░░░░░░   3,930
     4        4  █████████████████████████░░░░░░░░░░░░░░░░░   5,825
     5        5  ██████████████████████████████████░░░░░░░░   7,720
     6        5  ██████████████████████████████████████████   9,615

  trim
  turn  reports  context (tokens)
     1        1  █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░     140
     2        2  █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   2,035
     3        3  █████████████████░░░░░░░░░░░░░░░░░░░░░░░░░   3,930
     4        4  █████████████████░░░░░░░░░░░░░░░░░░░░░░░░░   3,965
     5        5  █████████████████░░░░░░░░░░░░░░░░░░░░░░░░░   4,000
     6        5  ██████████████████░░░░░░░░░░░░░░░░░░░░░░░░   4,035
  → 3 compaction(s); cleared/condensed 6 tool result(s)

Peak context: 9,615 → 4,035 tokens (58% 

### Under the hood — what trim does to the messages

Trim is pure, deterministic, and inspectable. Here we run it directly on a synthetic 5-report conversation: the **last 2** results are kept verbatim, the older ones become a placeholder. The original payloads stay on the edit record, so nothing is truly lost.

In [6]:
def tool_pair(call_id, text):
    use = AssistantMessagePart(parts=[ToolUsePart(id=call_id, name="fetch_report", inputs={})])
    result = UserMessagePart(
        parts=[
            ToolResultPart(tool_use_id=call_id, name="fetch_report", parts=[TextPart(text=text)])
        ]
    )
    return use, result


demo = []
for i in range(5):
    demo.extend(tool_pair(f"call-{i}", text=f"large report payload #{i} " * 10))

before = estimate_input_tokens(demo)
trimmed, applied = edit_context(demo, ContextPolicy(keep_last_n=2))
after = estimate_input_tokens(trimmed)
print(
    f"context: {before:,} → {after:,} tokens   (kept last 2, cleared {applied.cleared_tool_uses})\n"
)
for msg in trimmed:
    if isinstance(msg, UserMessagePart) and isinstance(msg.parts[0], ToolResultPart):
        p = msg.parts[0]
        print(f"  {p.tool_use_id}: {p.parts[0].text[:54]}")

context: 317 → 166 tokens   (kept last 2, cleared 3)

  call-0: [Tool result cleared to save context.]
  call-1: [Tool result cleared to save context.]
  call-2: [Tool result cleared to save context.]
  call-3: large report payload #3 large report payload #3 large 
  call-4: large report payload #4 large report payload #4 large 


## Step 3 — Summarize: keep the meaning, drop the bulk

Trim discards the old payloads. When you need to *remember* what they said, use `mode="summarize"`: Dobby makes one LLM call to digest the old span, then replaces the span with that summary. Below are the **real digests** the model produced during the run.

In [7]:
SUMM = ContextPolicy(context_window=2000, trigger_pct=0.5, keep_last_n=2, mode="summarize")
summarize = await run_agent("summarize", policy=SUMM)
compare(baseline, summarize)
print("\nDigests the model wrote in place of the old reports:")
for e in summarize.edits:
    for ae in e.applied_edits:
        if ae.summary_text:
            print("  ▸", " ".join(ae.summary_text.split())[:150], "…")

  baseline
  turn  reports  context (tokens)
     1        1  █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░     140
     2        2  █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   2,035
     3        3  █████████████████░░░░░░░░░░░░░░░░░░░░░░░░░   3,930
     4        4  █████████████████████████░░░░░░░░░░░░░░░░░   5,825
     5        5  ██████████████████████████████████░░░░░░░░   7,720
     6        5  ██████████████████████████████████████████   9,615

  summarize
  turn  reports  context (tokens)
     1        1  █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░     140
     2        2  █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   2,035
     3        3  █████████████████░░░░░░░░░░░░░░░░░░░░░░░░░   3,930
     4        4  ██████████████████░░░░░░░░░░░░░░░░░░░░░░░░   4,068
     5        5  ██████████████████░░░░░░░░░░░░░░░░░░░░░░░░   4,187
     6        5  ███████████████████░░░░░░░░░░░░░░░░░░░░░░░   4,336
  → 3 compaction(s); cleared/condensed 3 tool result(s)

Digests the model wrote in place of

## Step 4 — Let the agent compact itself

You can hand the agent a `CompactContextTool`. Now the **model decides** when to compact and writes its own instructions for what to preserve — full agent-native parity with the automatic policy.

In [8]:
# The auto-policy window is set high (100k) so the automatic trigger never fires here —
# every compaction below is one the *agent itself* requested via the tool.
COMPACT_PROMPT = (
    "You are a research assistant. You MUST fetch a report for EVERY requested topic "
    "before writing any summary. Call fetch_report for exactly ONE topic per turn. "
    "Immediately after the THIRD report comes back, call compact_context exactly once "
    "(with instructions to preserve the key findings), then continue fetching the "
    "remaining topics. Do not stop until all topics have been fetched. "
    f"Topics: {', '.join(TOPICS)}."
)
ct = await run_agent(
    "agent-invoked",
    policy=ContextPolicy(context_window=100_000, trigger_pct=0.8, keep_last_n=2, mode="summarize"),
    tools=[FetchReportTool(), CompactContextTool()],
    system_prompt=COMPACT_PROMPT,
    max_iterations=14,
)
print(
    f"The agent fetched {max(ct.concepts)} reports and chose to compact {len(ct.edits)} time(s):"
)
for e in ct.edits:
    print("  ▸ compaction:", [ae.type for ae in e.applied_edits])

The agent fetched 5 reports and chose to compact 1 time(s):
  ▸ compaction: ['summarize']


## The bottom line

Same agent. Same five reports fetched. The only thing that changed is the compaction policy.

In [9]:
show_dashboard([baseline, trim, summarize])

strategy      reports  peak context  vs baseline  extra LLM calls
  ---------------------------------------------------------------
baseline            5         9,615            —                0
trim                5         4,035         -58%                0
summarize           5         4,336         -55%                3
